# Ranking: Feature Engineering

In [1]:
import sys
import os
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath("../src"))

## Spark + Data Initialisation
Initialising Spark and loading data saved as parquets from retrieval stage

In [2]:
from utils.spark_session import get_spark

spark = get_spark()
spark.sparkContext.setLogLevel("INFO")

In [3]:
# Reloading data from ALS retrieval stage:
# recommendations per user from ALS
# test data

als_candidates = spark.read.parquet('../data/retrieval/als_candidates.parquet')
test = spark.read.parquet('../data/retrieval/test_filtered.parquet')
train = spark.read.parquet('../data/retrieval/train.parquet')

## User Features

In [4]:
# Dataframe for quick development
import configs.settings as cfg
user_sample = train.select(cfg.USER_COL).distinct().sample(0.01, seed=10)

train_sample = train.join(user_sample, on=cfg.USER_COL, how='inner')
train_sample.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|    68|    150|   3.0|1996-05-02 15:30:43|
|    68|    296|   4.0|1996-05-02 15:30:43|
|    68|    590|   5.0|1996-05-02 15:30:43|
|    68|    592|   3.0|1996-05-02 15:30:43|
|    68|     10|   4.0|1996-05-02 15:31:26|
+------+-------+------+-------------------+
only showing top 5 rows


In [43]:
train.orderBy('userId').show(150)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|    924|   3.5|2004-09-10 03:06:38|
|     1|    919|   3.5|2004-09-10 03:07:01|
|     1|   2683|   3.5|2004-09-10 03:07:30|
|     1|   1584|   3.5|2004-09-10 03:07:36|
|     1|   1079|   4.0|2004-09-10 03:07:45|
|     1|    653|   3.0|2004-09-10 03:08:11|
|     1|   2959|   4.0|2004-09-10 03:08:18|
|     1|    337|   3.5|2004-09-10 03:08:29|
|     1|   1304|   3.0|2004-09-10 03:08:40|
|     1|   3996|   4.0|2004-09-10 03:08:47|
|     1|    151|   4.0|2004-09-10 03:08:54|
|     1|    112|   3.5|2004-09-10 03:09:00|
|     1|   1374|   4.0|2004-09-10 03:09:06|
|     1|   1246|   3.5|2004-09-10 03:09:19|
|     1|   1370|   3.0|2004-09-10 03:09:24|
|     1|   2291|   4.0|2004-09-10 03:09:37|
|     1|   4306|   4.0|2004-09-10 03:09:44|
|     1|   1214|   4.0|2004-09-10 03:12:57|
|     1|   1278|   4.0|2004-09-10 03:13:06|
|     1|   1219|   4.0|2004-09-1

In [7]:
from pyspark.sql import functions as F
als_candidates.groupBy('userId').agg(
    F.count(F.col('rating')).alias('num_rat')
).orderBy('num_rat').show(10)

+------+-------+
|userId|num_rat|
+------+-------+
| 83090|     58|
|118205|     69|
| 34576|     90|
|    65|    100|
|   458|    100|
|   879|    100|
|   883|    100|
|  1223|    100|
|  1977|    100|
|  2096|    100|
+------+-------+
only showing top 10 rows


The following users have < 100 ratings due to duplicate removals:
| User | Rating |
| 83090|         58|
|118205|         69|
| 34576|         90|

### Build User Features

In [8]:
from data.stats import compute_global_std

global_std = compute_global_std(train_sample)
global_std

1.085693831580545

In [9]:
from features.user_features import build_user_features

user_feature = build_user_features(train_sample, global_std, k_shrinkage=20)
user_feature.show(5)

+------+------------------+-----------------+------------------+------------------+------------------------+------------------+
|userId|   user_avg_rating|user_rating_count|   user_rating_std|  log_rating_count|days_since_last_activity|    user_bayes_std|
+------+------------------+-----------------+------------------+------------------+------------------------+------------------+
| 33890| 4.363157894736842|               95|0.6974196562556085| 4.564348191467836|                     897|0.7649455997903801|
| 39712| 3.849462365591398|               93|0.8333567084145663| 4.543294782270004|                       7|0.8780181461430581|
| 44618| 4.102564102564102|               39|0.8206182455888074|3.6888794541139363|                    4637|0.9104743764334642|
| 54718|            4.6875|               40|0.5510770340305404| 3.713572066704308|                    1098|0.7292826332138753|
| 65178|3.4473684210526314|               38|0.8913205635931025|3.6635616461296463|                    6

## Item Features

In [10]:
from data.stats import compute_global_mean

mu = compute_global_mean(train_sample)
mu

3.514757490313812

In [11]:
# build out item features
from features.item_features import build_item_features

item_feature = build_item_features(train_sample, mu, C=50)
item_feature.show(5)

+-------+------------------+-----------------+------------------+------------------+
|movieId|   item_avg_rating|item_rating_count| item_bayesian_avg|  log_rating_count|
+-------+------------------+-----------------+------------------+------------------+
|   1580| 3.521885521885522|              297|3.5208584279991086| 5.697093486505405|
|   3918|              3.25|               12| 3.463514105091784|2.5649493574615367|
|   2366|3.7142857142857144|               49|3.6135138839968746| 3.912023005428146|
|   1645|              3.42|               75|3.4579029961255245| 4.330733340286331|
|   2122|               2.3|               15|3.2344288387029323| 2.772588722239781|
+-------+------------------+-----------------+------------------+------------------+
only showing top 5 rows


## Biases

In [12]:
# Hyperparameters
bias_hparams = {
    'reg_param': 10,
    'tau': 3,
    'epsilon': 1e-6
}

### Item Biases

In [13]:
from features.biases import compute_item_bias

item_bias = compute_item_bias(item_feature, mu=mu, reg_param=bias_hparams['reg_param'])
item_bias.show(5)

+-------+--------------------+
|movieId|           item_bias|
+-------+--------------------+
|   1580|0.006895848132892023|
|   3918|-0.14441317653480662|
|   2366| 0.16570988092581718|
|   1645|-0.08360955027689311|
|   2122| -0.7288544941882874|
+-------+--------------------+
only showing top 5 rows


### User Biases

In [14]:
from features.biases import compute_user_bias

user_bias = compute_user_bias(train_sample, user_feature, item_bias, mu=mu, reg_param=bias_hparams['reg_param'])
user_bias.show(5)

+------+-------------------+
|userId|          user_bias|
+------+-------------------+
|  4818|-0.0989333976897965|
| 48875|0.04997686402348942|
| 69637|0.12752048809844224|
|127444|0.06631286807157048|
|  6357|0.28574937707858467|
+------+-------------------+
only showing top 5 rows


## Residuals & Weighting

### Expected Rating

In [15]:
from features.biases import compute_expected_rating

expected_rating = compute_expected_rating(train_sample, user_bias, item_bias, mu)
expected_rating.show(5)

+-------+------+------+-------------------+--------------------+------------------+
|movieId|userId|rating|          user_bias|           item_bias|   expected_rating|
+-------+------+------+-------------------+--------------------+------------------+
|   4896|  4818|   4.0|-0.0989333976897965| 0.11793104279248938|3.5337551354165053|
|   3052|  4818|   4.0|-0.0989333976897965|  0.2094546243257119| 3.625278716949728|
|    350|  4818|   4.0|-0.0989333976897965|-0.04114396357998...|3.3746801290440267|
|   2692|  4818|   4.0|-0.0989333976897965|  0.5095596757071089|3.9253837683311246|
|   1080|  4818|   3.0|-0.0989333976897965| 0.45393654131933703| 3.869760633943353|
+-------+------+------+-------------------+--------------------+------------------+
only showing top 5 rows


## Weightings

In [16]:
from features.biases import compute_user_weights

weights = compute_user_weights(
    expected_rating, user_feature, tau=bias_hparams['tau'],epsilon=bias_hparams['epsilon'])
weights.show(5)

+------+-------+--------------------+
|userId|movieId|              weight|
+------+-------+--------------------+
|  4818|   4896| 0.16944201448439195|
|  4818|   3052| 0.13664668737393296|
|  4818|    350|  0.2255214983427262|
|  4818|   2692|0.027374157755182905|
|  4818|   1080| -0.3087520169887564|
+------+-------+--------------------+
only showing top 5 rows


## Tag Features

### PCA dimension reduction

In [17]:
import configs.settings as cfg
from features.tag_features import build_genome_pca_features
# load csv as dataframe
genomes = spark.read.csv("../data/raw/genome_scores.csv", schema=cfg.GENOMIC_SCHEMA, header=True)

# conduct pca and scaling on genome features
genome_scalar_model, genome_pca_df, genome_pca_model =  (
    build_genome_pca_features(genomes, k=45)
)

In [6]:
# Saving genome pca dataframe
genome_pca_df.write.mode('overwrite').parquet('../data/features/genome_pca_45.parquet')

### Normalise Tags

In [18]:
from data.preprocessing import normaliser
item_tag_norm = normaliser(genome_pca_df, input_col='pca_features', output_col='item_tag_norm')
item_tag_norm.show(5)

+-------+--------------------+
|movieId|       item_tag_norm|
+-------+--------------------+
|    496|[0.07888750859851...|
|    148|[-0.0275135375140...|
|    463|[-0.6914273308849...|
|    471|[0.40777863452039...|
|    833|[-0.6868075439068...|
+-------+--------------------+
only showing top 5 rows


## User Vector

In [19]:
genome_pca_df.cache()
genome_pca_df.show(5)

+-------+--------------------+
|movieId|        pca_features|
+-------+--------------------+
|    496|[0.20513351766232...|
|    148|[-0.0423306077345...|
|    463|[-1.6947855038298...|
|    471|[1.49517609789605...|
|    833|[-2.2212663724634...|
+-------+--------------------+
only showing top 5 rows


In [20]:
from features.user_vectors import build_user_tags

user_tags = build_user_tags(weights, genome_pca_df)
user_tags.show(5)

+------+--------------------+
|userId|            user_tag|
+------+--------------------+
|     2|[-0.0090591637174...|
|    58|[0.41750818418013...|
|   198|[0.58639814527679...|
|   549|[0.53674413726954...|
|   621|[0.65423291370568...|
+------+--------------------+
only showing top 5 rows


In [22]:
from data.preprocessing import normaliser

user_tag_norm = normaliser(user_tags, input_col='user_tag', output_col='user_tag_norm')
user_tag_norm.show(5)

+------+--------------------+
|userId|       user_tag_norm|
+------+--------------------+
|     2|[-0.0090012420972...|
|    58|[0.76706879586112...|
|   198|[0.57380713104192...|
|   549|[0.71438968188835...|
|   621|[0.71647857193433...|
+------+--------------------+
only showing top 5 rows


## Cosine Similarity
Compare user average tag (genome scores) to films

In [27]:
from ranking.tag_similarity import compute_tag_similarity

similarity_score = compute_tag_similarity(user_tag_norm, item_tag_norm, als_candidates)

similarity_score.orderBy('userId').show(150)

+------+-------+----------+--------------------+
|userId|movieId| als_score|          similarity|
+------+-------+----------+--------------------+
|     2|   3984| 0.9485389|  0.4224089930053676|
|     2|   3932| 0.7799852|  0.3095395921548146|
|     2|   2160| 0.7706593|-0.02358147540604109|
|     2|   4008|0.73098165| 0.16088886091620883|
|     2|   2792|0.70366883|-0.06181990831171765|
|     2|   3624| 0.6887915| 0.36628272553198415|
|     2|    780| 0.6784826|  0.7735895394285974|
|     2|    260|  0.675555|  0.6341979171383008|
|     2|   3755|0.66997963| 0.47677004240862014|
|     2|    524| 0.6596142| 0.26316417315246743|
|     2|    608|0.64309454|-0.07982620766517762|
|     2|   2657|0.63856006|-0.18398318806588917|
|     2|    648| 0.6136334|  0.5607935577361072|
|     2|    366| 0.6078684|-0.11444831954852064|
|     2|      1|0.60410976| 0.38485074060435015|
|     2|   3535|0.59444904|-0.19564076467284489|
|     2|   1035|0.57982844| 0.13670921843711512|
|     2|    356|0.57